# Beyond Words — Phase 5: ASR transcription of every clip (Kaggle)

Runs `transcribe.py` over all 9,303 clips with each off-the-shelf Bangla ASR front-end.
No model is fine-tuned. One CSV per front-end (`relpath,district,text`) lands in `/kaggle/working`.

**Before running — notebook settings (right sidebar):**
1. **Accelerator**: GPU T4 (not P100)
2. **Internet**: ON (needed to clone the repo and download the models)
3. **Input**: attach your private dataset `bangla-accent-voice-data`

Each run is resume-safe: an interrupted front-end continues from its existing CSV.

In [ ]:
!git clone --depth 1 https://github.com/towfique-elahe/beyond-words.git /tmp/beyond-words
!nvidia-smi -L

In [ ]:
# (tag, HF model id, backend, extra args). Hishab goes last: installing NeMo can
# change the torch/transformers versions the other front-ends were run with.
MODELS = [
    ('tugstugi_ben10', 'bengaliAI/tugstugi_bengaliai-regional-asr_whisper-medium', 'whisper',
     '--gen-config-from openai/whisper-medium'),
    ('tugstugi', 'bengaliAI/tugstugi_bengaliai-asr_whisper-medium', 'whisper',
     '--gen-config-from openai/whisper-medium'),
    ('whisper_large_v3', 'openai/whisper-large-v3', 'whisper', ''),
    ('w2v_xlsr_bn', 'arijitx/wav2vec2-xls-r-300m-bengali', 'ctc', ''),
    ('hishab_fastconformer', 'hishab/hishab_bn_fastconformer', 'nemo', ''),
]

# Locate the audio root inside the attached dataset (the folder holding Barishal/, Formal/, ...)
from pathlib import Path

AUDIO_ROOT = None
for p in Path('/kaggle/input').rglob('Barishal'):
    if p.is_dir():
        AUDIO_ROOT = str(p.parent)
        break
assert AUDIO_ROOT, 'Dataset not found — attach bangla-accent-voice-data as input'
print('AUDIO_ROOT =', AUDIO_ROOT)

In [ ]:
import time

for tag, model, backend, extra in MODELS:
    if backend == 'nemo':
        !pip install -q "nemo_toolkit[asr]"
    t0 = time.time()
    !python /tmp/beyond-words/transcribe.py \
        --model {model} --backend {backend} --tag {tag} {extra} \
        --manifest /tmp/beyond-words/splits/manifest_split_strict_rel.csv \
        --audio-root "{AUDIO_ROOT}" --out-dir /kaggle/working --batch 16
    print(f'{tag}: {(time.time() - t0) / 60:.1f} min')

In [ ]:
import pandas as pd

for tag, *_ in MODELS:
    f = Path('/kaggle/working') / f'{tag}.csv'
    if not f.exists():
        print(f'{tag}: MISSING')
        continue
    df = pd.read_csv(f)
    print(f'{tag}: {len(df)} rows, {df.text.isna().sum()} empty')
    print(df.groupby('district').head(1)[['district', 'text']].to_string(index=False), '\n')

When it finishes, download the CSVs from the Output tab (or `kaggle kernels output`)
into the local repo's `transcripts/` folder for `build_lexicon.py` and `lexical_did.py`.